# MojoVec 0.6.1 — Python Quickstart

Этот ноутбук показывает актуальный managed Python API MojoVec:

- создание Flat или SQ8-коллекции;
- пакетное добавление векторов, metadata и documents;
- vector search с Chroma-style `where`;
- BM25 и hybrid search через RRF;
- `upsert`, soft delete, статистику и compaction;
- атомарное сохранение и загрузку через mmap.

MojoVec работает внутри процесса: сервер, ручное выделение памяти и вызовы `free()` не нужны. ID коллекции в текущей версии имеют тип `int`.

## 1. Установка

Устанавливаем именно `0.6.1` в окружение текущего kernel. Если в kernel уже был импортирован предыдущий релиз, установочная ячейка автоматически запросит его перезапуск; после этого выполните notebook ещё раз.

In [ ]:
%pip install --upgrade "mojovec==0.6.1"

from importlib.metadata import version as _distribution_version
import sys as _sys

_installed = _distribution_version("mojovec")
_loaded = _sys.modules.get("mojovec")
_loaded_version = getattr(_loaded, "__version__", None)
if _loaded_version is not None and _loaded_version != _installed:
    print(
        f"Перезапускаю kernel: в памяти MojoVec {_loaded_version}, "
        f"на диске {_installed}. После перезапуска выполните notebook ещё раз."
    )
    get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -n 1


In [ ]:
!lscpu


In [ ]:
from importlib.metadata import version as distribution_version
from pathlib import Path
import ctypes
import site
import sys

expected_version = "0.6.1"
installed_version = distribution_version("mojovec")
loaded_module = sys.modules.get("mojovec")
loaded_version = getattr(loaded_module, "__version__", None)

if loaded_version is not None and loaded_version != installed_version:
    raise RuntimeError(
        f"В kernel загружен MojoVec {loaded_version}, а pip установил "
        f"{installed_version}. Перезапустите kernel и выполните ячейки снова."
    )
if installed_version != expected_version:
    raise RuntimeError(
        f"Ожидался MojoVec {expected_version}, установлен {installed_version}. "
        "Повторно выполните ячейку установки."
    )

def preload_mojo_runtime():
    suffix = ".dylib" if sys.platform == "darwin" else ".so"
    directories = [Path.home() / ".pixi" / "envs" / "mojo" / "lib"]
    directories.extend(Path(path) / "modular" / "lib" for path in site.getsitepackages())
    names = [
        "libMSupportGlobals",
        "libAsyncRTRuntimeGlobals",
        "libAsyncRTMojoBindings",
        "libKGENCompilerRTShared",
    ]
    for directory in directories:
        if not (directory / f"libKGENCompilerRTShared{suffix}").is_file():
            continue
        for name in names:
            library = directory / f"{name}{suffix}"
            if library.is_file():
                ctypes.CDLL(str(library), mode=ctypes.RTLD_GLOBAL)
        return True
    return False

try:
    import mojovec
except ImportError:
    # Fallback для окружений с отдельно установленным Mojo runtime.
    if not preload_mojo_runtime():
        raise
    import mojovec

print("MojoVec version:", mojovec.__version__)

## 2. Создание коллекции

`quantized=True` включает SQ8. Для Flat Float32 достаточно поставить `False`; остальной API не меняется.

Метрики:

- `l2` — squared Euclidean distance;
- `cosine` — `1 - cosine_similarity`;
- `ip` — `1 - inner_product`.

Во всех случаях меньшее расстояние означает более близкий результат.

In [ ]:
collection = mojovec.Collection(
    dimension=4,
    M=16,
    ef_construction=96,
    ef_search=64,
    quantized=True,
    name="knowledge_base",
    metric="cosine",
)

collection

## 3. Batch add: vectors + metadata + documents

Python API принимает как вложенные векторы `[[...], [...]]`, так и один flattened row-major список. Metadata поддерживает скалярные `str`, `int`, `float` и `bool`.

In [ ]:
collection.add(
    ids=[101, 202, 303, 404],
    embeddings=[
        [1.0, 0.0, 0.0, 0.0],
        [0.9, 0.1, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
    ],
    metadatas=[
        {"category": "guide", "year": 2024, "published": True},
        {"category": "internals", "year": 2026, "published": True},
        {"category": "release", "year": 2026, "published": False},
        {"category": "tutorial", "year": 2027, "published": True},
    ],
    documents=[
        "Vector search introduction",
        "HNSW graph traversal and vector search",
        "Database release notes",
        "BM25 full text search tutorial",
    ],
)

print(collection.stats())
print(collection.get_metadata(202))
print(collection.get_document(202))

## 4. Vector search и `where`

`where` поддерживает `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or` и `$not`.

Каждый managed query возвращает пять ключей: `ids`, `distances`, `metadatas`, `documents`, `scores`. Vector search заполняет `distances`; BM25 и hybrid заполняют `scores`.

In [ ]:
vector_result = collection.query(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    n_results=3,
    where={
        "$and": [
            {"published": True},
            {"year": {"$gte": 2024}},
            {"category": {"$in": ["guide", "internals", "tutorial"]}},
        ]
    },
)

vector_result

In [ ]:
def show_results(result):
    for query_index, row_ids in enumerate(result["ids"]):
        print(f"query {query_index}")
        for rank, record_id in enumerate(row_ids):
            if record_id < 0:  # padding, если совпадений меньше n_results
                continue
            value = (
                result["distances"][query_index][rank]
                if result["distances"]
                else result["scores"][query_index][rank]
            )
            print(
                f"  rank={rank + 1} id={record_id} value={value:.6f}\n"
                f"    metadata={result['metadatas'][query_index][rank]}\n"
                f"    document={result['documents'][query_index][rank]}"
            )


show_results(vector_result)

## 5. BM25 document search

Передача `query_texts` выбирает BM25. Анализатор применяет Unicode lowercase, word boundaries и встроенные английские/русские stopwords без stemming.

In [ ]:
bm25_result = collection.query(
    query_texts=["HNSW vector search"],
    n_results=3,
    where={"published": True},
)

show_results(bm25_result)

## 6. Hybrid search через RRF

Hybrid search объединяет ранги HNSW и BM25 через reciprocal rank fusion. Embedding и текст с одинаковым индексом образуют одну hybrid query.

In [ ]:
hybrid_result = collection.query_hybrid(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    query_texts=["HNSW graph traversal"],
    n_results=3,
    rrf_k=60,
    candidate_multiplier=4,
    where={"published": True},
)

show_results(hybrid_result)

## 7. Update, upsert, delete и compaction

- `add` принимает только новые ID;
- `upsert` вставляет отсутствующие ID и заменяет существующие;
- `update` требует, чтобы все ID существовали;
- `delete` выполняет soft delete и игнорирует неизвестные ID.

Vector-only update сохраняет прежние metadata/documents. Если payload передан явно, соответствующий payload заменяется целиком.

In [ ]:
collection.upsert(
    ids=[202, 505],
    embeddings=[
        [0.85, 0.15, 0.0, 0.0],
        [0.0, 0.0, 0.9, 0.1],
    ],
    metadatas=[
        {"category": "internals", "year": 2028, "published": True},
        {"category": "guide", "year": 2028, "published": True},
    ],
    documents=[
        "Updated HNSW internals",
        "Persistence and mmap guide",
    ],
)
collection.delete([303])

print("before compaction:", collection.stats())
report = collection.compact_if_needed(deleted_ratio=0.20)
print("compaction report:", report)
print("after compaction:", collection.stats())

## 8. Атомарное сохранение и mmap load

`save()` публикует checksummed snapshot через атомарную замену файла. Для маленького учебного индекса ниже используется `mmap_threshold_bytes=0`, чтобы принудительно показать mmap-путь.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary_directory = TemporaryDirectory(prefix="mojovec-notebook-")
database_path = Path(temporary_directory.name) / "knowledge_base.mojovec"

collection.save(database_path)
loaded = mojovec.load(
    database_path,
    memory_mapped=True,
    mmap_threshold_bytes=0,
)

print("memory mapped:", loaded.is_memory_mapped())
print("loaded stats:", loaded.stats())
show_results(loaded.query([[1.0, 0.0, 0.0, 0.0]], n_results=2))

## Что дальше

- Для bulk NumPy-массивов используйте `upsert_numpy()` и `query_numpy()` с contiguous `int64` IDs и `float32` embeddings.
- Для crash recovery между snapshot-ами доступны `enable_wal()`, `flush_wal()`, `checkpoint()` и `mojovec.recover()`.
- Подбирайте `M`, `ef_construction` и `ef_search` по recall/latency на своих embeddings.